# Setup

In [ ]:
import requests, hashlib, shutil, os, json, socket, multiprocessing, configparser
from scipy.io import savemat, loadmat
from scipy import signal
from tqdm import tqdm
import numpy as np
import pandas as pd

settings_parser = configparser.ConfigParser()
settings_parser.read("../data/localsettings.ini")
MAIN_DATA_FOLDER = settings_parser.get("global", "data_path")
if not os.path.isdir(MAIN_DATA_FOLDER):
    os.mkdir(MAIN_DATA_FOLDER)
RESULTS_FOLDER = os.path.join(MAIN_DATA_FOLDER, "Results")
if not os.path.isdir(RESULTS_FOLDER):
    os.mkdir(RESULTS_FOLDER)
from mienc.support import surrogate

config_ini = os.path.join(MAIN_DATA_FOLDER, "config.ini")
block_size = 1024
config = configparser.ConfigParser()
seconds_for_chunk = 124

bands = {
    i: b
    for i, b in enumerate(
        [
            [1.0, 4.0],
            [4.0, 8.0],
            [8.0, 12.0],
            [12.0, 30.0],
            [30.0, 44.0],
        ],
        start=1,
    )
}
band_names = {
    i: b
    for i, b in enumerate(
        ["delta", "theta", "alpha", "beta", "gamma"],
        start=1,
    )
}


def fs_band(band):
    return int(bands[band][1] * 2 * 1.125 + 0.5)

In [ ]:
config.read(config_ini)
if config.has_section(socket.gethostname()):
    print("Config file already initialised.")
else:
    with open(config_ini, "a") as fp:
        print(
            "[global]\n"
            "bins = 0\n"
            "display = False\n"
            "workers = 2\n"
            "dataset = \n"
            "output_folder = .\n"
            "surrogates = 99\n"
            "\n\n"
            "[DEFAULT]\n"
            "dataset_sub = \n"
            "\n"
            "[correction]\n"
            "nsamples = 0\n"
            "steps = 200\n"
            "iters = 50000\n"
            "\n"
            "; ADD HERE THE INFO FOR SPECIFIC HOSTS\n"
            "; [hostname]\n"
            "; output_folder = path/to/output/directory/if/not/the/default/one\n"
            "; workers = # of workers to use\n"
            "\n"
            f"[{socket.gethostname()}]\n"
            f"output_folder = {RESULTS_FOLDER}\n"
            f"workers = {multiprocessing.cpu_count()-1}\n\n"
            "; ADD HERE THE INFO FOR NEW DATASETS\n"
            "; [dataset name]\n"
            "; file_path = path/to/mat/file/if/not/the/default/one\n"
            "; file_name = dataset.mat\n"
            "; field_name = dictionary key\n"
            "; healthy_control_start = 64\n"
            "; healthy_control_end = 0\n"
            '; A 0 here means "from the beginning" or "until the end"\n\n',
            file=fp,
        )

# fMRI

Download size: 6.8 GB

In [ ]:
# https://data.mendeley.com/datasets/crx7d22pym/1
# Kopal, Jakub; Hlinka, Jaroslav (2020), “Typicality of Functional Connectivity robustly captures motion artifacts in rs-fMRI across datasets, atlases, and preprocessing pipelines”, Mendeley Data, V1, doi: 10.17632/crx7d22pym.1
import zipfile

output_fMRI = os.path.join(MAIN_DATA_FOLDER, "fMRI_region_size")
if not os.path.isdir(output_fMRI):
    os.makedirs(output_fMRI)

sha256sum = hashlib.sha256()
sha256true = "7dd9b825d05c29e6bcc6f0c7caf66c52e25dcdb4bd853ee15a01ae3debe2530b"


if not os.path.isfile(os.path.join(output_fMRI, "crx7d22pym-1.zip")):
    eso_file_response = requests.get(
        f"https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/crx7d22pym-1.zip",
        stream=True,
    )
    with tqdm(
        desc="Dataset (Kopal, 2020)", unit="B", unit_scale=True, total=6768332232
    ) as progress_bar:
        with open(os.path.join(output_fMRI, "crx7d22pym-1.zip.tmp"), "wb") as fp:
            for data in eso_file_response.iter_content(block_size):
                progress_bar.update(len(data))
                sha256sum.update(data)
                fp.write(data)
    if sha256true == sha256sum.hexdigest():
        shutil.move(
            os.path.join(output_fMRI, "crx7d22pym-1.zip.tmp"),
            os.path.join(output_fMRI, "crx7d22pym-1.zip"),
        )
    else:
        print(f"Failed crx7d22pym-1.zip")
        os.unlink(os.path.join(output_fMRI, "crx7d22pym-1.zip.tmp"))

if not os.path.isfile(os.path.join(output_fMRI, "AAL_regions.xls")):
    aal_file_response = requests.get(
        f"https://plos.figshare.com/ndownloader/files/20702673"
    )
    with open(os.path.join(output_fMRI, "AAL_regions.xls"), "wb") as fp:
        for data in aal_file_response.iter_content(block_size):
            fp.write(data)

In [ ]:
regions = [
    10,
    30,
    50,
    70,
    100,
    150,
    200,
    230,
    270,
    300,
    350,
    400,
    450,
    500,
    550,
    600,
    650,
    700,
    750,
    800,
    850,
    900,
    950,
]

zip_file = zipfile.PyZipFile(os.path.join(output_fMRI, "crx7d22pym-1.zip"))
for n_regions in tqdm(regions, desc="# of regions"):
    if not os.path.isfile(os.path.join(output_fMRI, f"Main_cra_strin_{n_regions}.mat")):
        with open(
            os.path.join(output_fMRI, f"Main_cra_strin_{n_regions}.mat"), "wb"
        ) as fp:
            fp.write(zip_file.read(f"Data/Main_cra_strin_{n_regions}.mat"))

In [ ]:
bad_regions = {}
for n_regions in tqdm(regions, desc="Finding bad regions"):
    mat = loadmat(os.path.join(output_fMRI, f"Main_cra_strin_{n_regions}.mat"))[
        "subj_tcs"
    ]
    bad_regions[n_regions] = np.isclose(np.sum(np.abs(mat), axis=0), 0)

In [ ]:
def score(n_regions, remove):
    good = np.ones(bad_regions[n_regions].shape[1], dtype=bool)
    if remove:
        good[remove] = False
    filter = np.sum(no_sub := bad_regions[n_regions][:, good], axis=1) == 0
    left_ch = np.sum(filter)
    left_sub = np.sum(good)
    return (left_ch-1)*left_ch*left_sub/((filter.size-1)*filter.size*good.size)

In [ ]:
num_subj = bad_regions[10].shape[1]
baseline = np.mean([score(r, []) for r in regions])
single_remove = []
print("Fraction of the dataset retained:")
with tqdm(
    total=num_subj + 3 * (num_subj - 1) + 9 * (num_subj - 2) + 15 * (num_subj - 3),
    desc=f"All subjects best: [] - {baseline:.3}",
) as progress_bar:
    for i in range(num_subj):
        single_remove.append(([i], np.mean([score(r, [i]) for r in regions])))
        progress_bar.update()
    first_candidates = sorted(single_remove, key=lambda x: x[1], reverse=True)[:3]
    progress_bar.set_description(
        f"Removing 1 subj best {first_candidates[0][0]} - {first_candidates[0][1]:.3}"
    )
    double_remove = []
    for c in first_candidates:
        for i in range(num_subj):
            if i not in c[0]:
                double_remove.append(
                    ([*c[0], i], np.mean([score(r, [*c[0], i]) for r in regions]))
                )
                progress_bar.update()
    second_candidates = sorted(double_remove, key=lambda x: x[1], reverse=True)[:9]
    progress_bar.set_description(
        f"Removing 2 subj best {second_candidates[0][0]} - {second_candidates[0][1]:.3}"
    )
    triple_remove = []
    for c in second_candidates:
        for i in range(num_subj):
            if i not in c[0]:
                triple_remove.append(
                    ([*c[0], i], np.mean([score(r, [*c[0], i]) for r in regions]))
                )
                progress_bar.update()
    third_candidates = sorted(triple_remove, key=lambda x: x[1], reverse=True)[:15]
    progress_bar.set_description(
        f"Removing 3 subj best {third_candidates[0][0]} - {third_candidates[0][1]:.3}"
    )
    quadruple_remove = []
    for c in third_candidates:
        for i in range(num_subj):
            if i not in c[0]:
                quadruple_remove.append(
                    ([*c[0], i], np.mean([score(r, [*c[0], i]) for r in regions]))
                )
                progress_bar.update()
    fourth_candidates = sorted(quadruple_remove, key=lambda x: x[1], reverse=True)[:3]
    progress_bar.set_description(
        f"Removing 4 subj best {fourth_candidates[0][0]} - {fourth_candidates[0][1]:.3}"
    )

best = max(
    first_candidates + second_candidates + third_candidates + fourth_candidates,
    key=lambda x: x[1],
)
print(f"Overall best: {best[0]} - {best[1]:.4}")
bad_subjects = best[0]

In [ ]:
good = np.ones(bad_regions[regions[0]].shape[1], dtype=bool)
good[bad_subjects] = False
for n_regions in tqdm(regions, desc="Cleaning bad regions"):
    if not os.path.isfile(
        os.path.join(output_fMRI, f"clean_cra_strin_{n_regions}.mat")
    ):
        mat = loadmat(os.path.join(output_fMRI, f"Main_cra_strin_{n_regions}.mat"))[
            "subj_tcs"
        ]
        filter = np.sum(bad_regions[n_regions][:, good], axis=1) == 0
        savemat(
            os.path.join(output_fMRI, f"clean_cra_strin_{n_regions}.mat"),
            {"subj_tcs": mat[:, :, good][:, filter, :]},
        )

In [ ]:
config_string = f"[clean_cra]\nfile_path = {output_fMRI}\nfile_name = clean_cra_strin_%(dataset_sub)s.mat\nfield_name = subj_tcs\nhealthy_control_start = 0\nhealthy_control_end = 0\n"
config.read(config_ini)
if config.has_section("clean_cra"):
    print(
        f"The config file '{config_ini}' already contains section 'clean_cra'.\nCheck that it corresponds to:\n",
        config_string,
    )
else:
    with open(config_ini, "a") as fp:
        print(config_string, file=fp)
    print("'clean_cra' added to config file.")

# fMRI test-retest

Download size: 16 GB

In [ ]:
from glob import glob
from nilearn import datasets
from nilearn.maskers import NiftiLabelsMasker

output_fMRItr = os.path.join(MAIN_DATA_FOLDER, "fMRI_test_retest")
if not os.path.isdir(output_fMRItr):
    os.makedirs(output_fMRItr)


def getTree(datasetID, tag, tree=None):
    myurl = "https://openneuro.org/crn/graphql"
    header = {"content-type": "application/json"}
    if tree is not None:
        treepart = r"(tree: \"" + tree + r"\")"
    else:
        treepart = ""

    data = (
        r'{"query":"query snapshotFiles {snapshot(datasetId: \"'
        + datasetID
        + r"\", tag: \""
        + tag
        + r"\") {    files "
        + treepart
        + r'{id key filename size directory annexed}}}"}'
    )
    x = requests.post(myurl, data, headers=header)
    if x.status_code == 200:
        file_list = x.json()["data"]["snapshot"]["files"]
        return file_list
    else:
        print(f"Error while reading {tree=}")
        print(
            *[
                (
                    line
                    if len(line) < 96
                    else "\n".join([line[i : i + 96] for i in range(0, len(line), 96)])
                )
                for line in x.content.decode().splitlines()
            ],
            sep="\n",
        )
        return []


def getFile(datasetID, fileDict):
    if os.path.isfile(os.path.join(output_fMRItr, fileDict["filename"])):
        return
    if fileDict["filename"].endswith(("nii.gz", "set")):
        if not os.path.isfile(os.path.join(output_fMRItr, fileDict["filename"])):
            nii_file_response = requests.get(
                f'https://openneuro.org/crn/datasets/{datasetID}/objects/{fileDict["key"]}',
                stream=True,
            )

            md5true = fileDict["key"].split("--")[1].split(".")[0]
            md5sum = hashlib.md5()
            with tqdm(
                desc=fileDict["filename"], unit="B", unit_scale=True
            ) as progress_bar:
                with open(os.path.join(output_fMRItr, fileDict["key"]), "wb") as fp:
                    for data in nii_file_response.iter_content(block_size):
                        progress_bar.update(len(data))
                        md5sum.update(data)
                        fp.write(data)

            if md5true == md5sum.hexdigest():
                shutil.move(
                    os.path.join(output_fMRItr, fileDict["key"]),
                    os.path.join(output_fMRItr, fileDict["filename"]),
                )
            else:
                print(f"Failed {fileDict['filename']}")
                os.unlink(os.path.join(output_fMRItr, fileDict["key"]))

    elif fileDict["filename"].endswith(("json", "tsv")):
        if not os.path.isfile(os.path.join(output_fMRItr, fileDict["filename"])):
            nii_file_response = requests.get(
                f'https://openneuro.org/crn/datasets/{datasetID}/objects/{fileDict["key"]}'
            )
            print(os.path.join(output_fMRItr, fileDict["filename"]))
            with open(os.path.join(output_fMRItr, fileDict["filename"]), "wb") as fp:
                fp.write(nii_file_response.content)

In [ ]:
subject_dir = {
    file["filename"]: file["id"]
    for file in getTree("ds000221", "1.0.0")
    if file["directory"] and file["filename"].startswith("sub-")
}

prog_bar = tqdm(subject_dir, desc="Subject")
N = 0
for subj in prog_bar:
    sub_directories = {
        file["filename"]: file["id"]
        for file in getTree("ds000221", "1.0.0", subject_dir[subj])
    }
    if set(sub_directories.keys()) == {"ses-01", "ses-02"}:
        func_directories = []
        anat_directories = []
        for session in sub_directories:
            protocols = {
                file["filename"]: file["id"]
                for file in getTree("ds000221", "1.0.0", sub_directories[session])
            }
            try:
                func_directories.append(protocols["func"])
            except:
                # discarding subjects without test-retest
                tqdm.display(
                    f"Subject {subj} missing 'func' protocol in one session.", pos=-1
                )
                break
            if session == "ses-01":
                try:
                    anat_directories.append(protocols["anat"])
                except:
                    # discarding subjects with no anatomical data
                    prog_bar.display(
                        f"Subject {subj} missing 'anat' protocol in first session.",
                        pos=-1,
                    )
                    break
            if session == "ses-02":
                data_files = getTree("ds000221", "1.0.0", protocols["func"])
                file_for_TE = [
                    file
                    for file in data_files
                    if "ses-02" in file["filename"]
                    and file["filename"].endswith("json")
                ][0]
                nii_file_response = requests.get(
                    f'https://openneuro.org/crn/datasets/ds000221/objects/{file_for_TE["key"]}'
                )
                metadata_for_TE = json.loads(nii_file_response.content.decode())
                if not np.isclose(metadata_for_TE["EchoTime"], 0.03):
                    # discarding subjects with different TE across sessions
                    prog_bar.display(
                        f"Subject {subj} has different TE across sessions.\n", pos=-1
                    )
                    break

        else:
            N += 1
            prog_bar.display(f"Downloading subject {subj}.\n", pos=-1)
            for func in func_directories:
                data_files = getTree("ds000221", "1.0.0", func)
                for file in data_files:
                    if "rest" in file["filename"] and "acq-AP" in file["filename"]:
                        getFile("ds000221", file)
            for anat in anat_directories:
                data_files = getTree("ds000221", "1.0.0", anat)
                for file in data_files:
                    if "T1w" in file["filename"]:
                        getFile("ds000221", file)
    else:
        prog_bar.display(msg=f"Subject {subj} has only one session.\n", pos=-1)
print(
    "Add the following lines at the beginning of the matlab processing script `01a_V2V_fMRI_preprocessing.m`\n"
    f"directory= '{output_fMRItr}';\n"
    f"NSUBJECTS={N};\n"
)

Before continuing here, you should run the conn script `01a_V2V_fMRI_preprocessing.m` in this folder.

Adjust the first lines with the output at the end of the previous cell, add the path to `conn` and `spm12` if needed, then run:

```bash
matlab -nodisplay -nosplash -nodesktop -r "run('01a_V2V_fMRI_preprocessing.m');exit;"
```

In [ ]:
# first run `01a_V2V_fMRI_preprocessing.m` with Matlab (requires conn and spm12)
# matlab -nodisplay -nosplash -nodesktop -r "run('01a_V2V_fMRI_preprocessing.m');exit;"
sessions = [
    "ses-01_task-rest_acq-AP_run-01",
    "ses-02_task-rest_acq-AP_run-01",
    "ses-02_task-rest_acq-AP_run-02",
]


dataset = datasets.fetch_atlas_aal("SPM12")
atlas_filename = dataset.maps
masker = NiftiLabelsMasker(labels_img=atlas_filename, standardize=True)
for i, session in tqdm(enumerate(sessions), desc="Session", total=len(sessions)):
    if not os.path.isfile(
        os.path.join(output_fMRItr, f"lemon_aal_strin_90_ses-{i:02}.mat")
    ):
        files = sorted(
            glob(os.path.join(output_fMRItr, f"dswausub-*_{session}_bold.nii.gz"))
        )
        newmat = np.empty([657, 90, 14])
        for j, file in tqdm(
            enumerate(files), desc="Subject", total=len(files), leave=True
        ):
            time_series = masker.fit_transform(file)
            newmat[:, :, j] = time_series[:, :90]
        savemat(
            os.path.join(output_fMRItr, f"lemon_aal_strin_90_ses-{i:02}.mat"),
            {"subj_tcs": newmat},
        )


config_string = f"[lemon_aal]\nfile_path = {output_fMRItr}\nfile_name = lemon_aal_strin_90_ses-0%(dataset_sub)s.mat\nfield_name = subj_tcs\nhealthy_control_start = 0\nhealthy_control_end = 0\n"
config.read(config_ini)
if config.has_section("lemon_aal"):
    print(
        f"The config file '{config_ini}' already contains section 'lemon_all'.\nCheck that it corresponds to:\n",
        config_string,
    )
else:
    with open(config_ini, "a") as fp:
        print(config_string, file=fp)
    print("'lemon_aal' added to config file.")

# EEG

Download size: 14 GB

In [ ]:
# https://fcon_1000.projects.nitrc.org/indi/retro/MPI_LEMON/downloads/download_EEG.html
# Babayan, A.; Erbey, M.; Kumral, D.; Reinelt, J.; Reiter, A.; Röbbig, J.; Schaare, H. L.; Ragert, M.; Anwander, A.; Bazin, P.-L. ; Horstmann, A.; Lampe, L.; Nikulin, V. V.; Okon-Singer, H.; Preusser, S.; Pampel, A.; Rohr, C. S.; Sacher, J.; Thöne-Otto, A. I. T.; Trapp, S.; Nierhaus, T.; Altmann, D.; Arélin, K.; Blöchl, M.; Bongartz, E.; Breig, P.; Cesnaite, E.; Chen , S.; Cozatl, R.; Czerwonatis, S.; Dambrauskaite, G.; Paerisch, M.; Enders , J.; Engelhardt , M.; Fischer , M. M.; Forschack, N.; Golchert, J.; Golz, L.; Guran, C. A.; Hedrich, S.; Hentschel, N.; Hoffmann , D. I.; Huntenburg, J. M.; Jost , R.; Kosatschek, A.; Kunzendorf, S.; Lammers, H.; Lauckner, M.; Mahjoory, K.; Kanaan, A. S.; Mendes, N.; Menger, R.; Morino, E.; Naethe, K.; Neubauer , J.; Noyan, H.; Oligschläger, S.; Panczyszyn-Trzewik, P.; Poehlchen, D.; Putzke, N.; Roski, S.; Schaller , M.-C.; Schieferbein, A.; Schlaak, B.; Schmidt, R.; Gorgolewski, K. J.; Schmidt, H. M.; Schrimpf, A.; Stasch, S.; Voss, M.; Wiedemann, A.; Margulies, D. S.; Gaebler, M.; Villringer, A.: A mind-brain-body dataset of MRI, EEG, cognition, emotion, and peripheral physiology in young and old adults. Scientific Data 6 (2019)
import tarfile, mne, tempfile

missing_sub = [
    32335,
    32384,
    32404,
    32433,
    32437,
    32443,
    32445,
    32461,
    32488,
    32500,
    32519,
    32520,
    32527,
]
missing_aws = [
    32309,
    32365,
    32366,
    32374,
    32382,
    32410,
    32419,
    32426,
    32485,
    32486,
    32487,
    32489,
    32492,
]
bad_subjects = [
    32301,
    32306,
    32319,
    32323,
    32326,
    32329,
    32334,
    32336,
    32342,
    32343,
    32350,
    32354,
    32355,
    32357,
    32360,
    32362,
    32367,
    32368,
    32370,
    32372,
    32379,
    32380,
    32381,
    32383,
    32387,
    32396,
    32397,
    32399,
    32402,
    32417,
    32428,
    32434,
    32436,
    32439,
    32447,
    32449,
    32450,
    32466,
    32468,
    32469,
    32478,
    32493,
    32497,
    32498,
    32510,
    32512,
    32514,
    32521,
    32522,
    32523,
    32525,
    32526,
]
bad_electrodes = ["T7", "T8", "Cz", "F7", "CP6", "PO10", "Fp2"]
missing_ids = missing_sub + missing_aws + bad_subjects
first_id = 32301
last_id = 32528
tot_subjects = last_id - first_id - len(missing_ids) + 1

output_EEG = os.path.join(MAIN_DATA_FOLDER, "EEG")

tmp_dir = os.path.join(output_EEG, f"temporary_arrays")
if not os.path.isdir(tmp_dir):
    os.mkdir(tmp_dir)
    
mne.set_log_level(verbose="error")

In [ ]:
with requests.Session() as session, tqdm(
    total=tot_subjects, desc="Subjects"
) as main_pbar:
    for sid in range(first_id, last_id + 1):
        if sid in missing_ids:
            continue
        if not os.path.isfile(os.path.join(output_EEG, f"sub-{sid:06}.tar.gz")):
            try:
                lemon_file_response = session.get(
                    f"https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/MPI-LEMON/Compressed_tar/EEG_MPILMBB_LEMON/EEG_Preprocessed_BIDS_ID/EEG_Preprocessed/sub-{sid:06}.tar.gz",
                    stream=True,
                )
                file_size = int(lemon_file_response.headers["Content-Length"])
                with tqdm(
                    desc=f"Subject {sid}",
                    unit="B",
                    unit_scale=True,
                    total=file_size,
                    leave=False,
                ) as progress_bar:
                    with open(
                        os.path.join(output_EEG, f"sub-{sid:06}.tar.gz.tmp"), "wb"
                    ) as fp:
                        for data in lemon_file_response.iter_content(block_size):
                            progress_bar.update(len(data))
                            fp.write(data)

                assert (
                    file_size
                    == os.stat(
                        os.path.join(output_EEG, f"sub-{sid:06}.tar.gz.tmp")
                    ).st_size
                )
                os.rename(
                    os.path.join(output_EEG, f"sub-{sid:06}.tar.gz.tmp"),
                    os.path.join(output_EEG, f"sub-{sid:06}.tar.gz"),
                )
            except Exception as e:
                if os.path.isfile(os.path.join(output_EEG, f"sub-{sid:06}.tar.gz.tmp")):
                    os.unlink(os.path.join(output_EEG, f"sub-{sid:06}.tar.gz.tmp"))
                print(e)

        main_pbar.update()


In [ ]:
rs = np.random.default_rng(42)
raw = None
with tqdm(total=tot_subjects, desc="Subjects") as main_pbar:
    for sid in range(first_id, last_id + 1):
        if sid in missing_ids:
            continue

        if os.path.isfile(os.path.join(tmp_dir, f"shadow_sub{sid:06}_band5.npy")):
            main_pbar.update()
            continue

        try:
            dirpath = tempfile.mkdtemp()
            with tarfile.open(
                os.path.join(output_EEG, f"sub-{sid:06}.tar.gz"), "r:gz"
            ) as tar:
                tar.extractall(dirpath)
            set_file = os.path.join(dirpath, f"sub-{sid:06}", f"sub-{sid:06}_EC.set")

            raw = mne.io.read_raw_eeglab(set_file).drop_channels(
                bad_electrodes, on_missing="ignore"
            )
            boundaries = mne.events_from_annotations(
                raw, regexp="boundary", verbose=False
            )[0][:, 0]

            fs_raw = raw.info["sfreq"]
            for band in bands:
                filtered = []
                power_pieces = []
                shadow_pieces = []
                i_old = 0
                for i in boundaries:
                    if i - i_old < 1250:
                        # discarding segments < 5 s
                        i_old = i
                        continue
                    tmp = raw.copy().crop(i_old / fs_raw, i / fs_raw)
                    tmp.load_data(verbose=False)
                    tmp.filter(
                        l_freq=bands[band][0],
                        h_freq=bands[band][1],
                        method="iir",
                        phase="zero",
                        verbose=False,
                    )

                    tmp_data = tmp.resample(200)._data.T
                    samples = tmp_data.shape[0] // 25

                    power = np.absolute(signal.hilbert(tmp_data, axis=0))
                    dsp = np.average(
                        np.reshape(power[: samples * 25], (-1, 25, power.shape[1])), 1
                    )
                    power_pieces.append(dsp.copy())

                    shadow_data = surrogate(tmp_data, random_state=rs)
                    shadow_power = np.absolute(signal.hilbert(shadow_data, axis=0))
                    dsp = np.average(
                        np.reshape(
                            shadow_power[: samples * 25],
                            (-1, 25, shadow_power.shape[1]),
                        ),
                        1,
                    )
                    shadow_pieces.append(dsp.copy())

                    tmp.resample(fs_band(band))
                    filtered.append(tmp.copy())
                    i_old = i
                filtered[0].append(filtered[1:])
                np.save(
                    os.path.join(tmp_dir, f"filtered_sub{sid:06}_band{band}.npy"),
                    filtered[0]._data.T,
                )
                np.save(
                    os.path.join(tmp_dir, f"power_sub{sid:06}_band{band}.npy"),
                    np.concatenate(power_pieces),
                )
                np.save(
                    os.path.join(tmp_dir, f"shadow_sub{sid:06}_band{band}.npy"),
                    np.concatenate(shadow_pieces),
                )

        finally:
            shutil.rmtree(dirpath)
        main_pbar.update()
if not os.path.isfile(os.path.join(output_EEG, "good_electrodes.json")):
    if raw is None:
        try:
            dirpath = tempfile.mkdtemp()
            with tarfile.open(
                os.path.join(output_EEG, f"sub-{sid:06}.tar.gz"), "r:gz"
            ) as tar:
                tar.extractall(dirpath)
            set_file = os.path.join(dirpath, f"sub-{sid:06}", f"sub-{sid:06}_EC.set")

            raw = mne.io.read_raw_eeglab(set_file).drop_channels(
                bad_electrodes, on_missing="ignore"
            )
        finally:
            shutil.rmtree(dirpath)
    with open(os.path.join(output_EEG, "good_electrodes.json"), "w") as fp:
        json.dump(raw.ch_names, fp)

In [ ]:

for out_name, tmp_name in [
    ("EEG_124s", "filtered"),
    ("electrodeBLP", "power"),
    ("electrodeBLP_shadow", "shadow"),
]:
    for band in bands:
        fs = 8 if "BLP" in out_name else fs_band(band)
        samples_for_chunk = seconds_for_chunk * fs
        data = np.empty([samples_for_chunk * 3, 54, tot_subjects])
        sn = 0
        for sid in range(first_id, last_id + 1):
            if sid in missing_ids:
                continue
            try:
                data[:, :, sn] = np.load(
                    os.path.join(tmp_dir, f"{tmp_name}_sub{sid:06}_band{band}.npy")
                )[: samples_for_chunk * 3, :]
            except:
                print(sid)
            sn += 1
        savemat(
            os.path.join(output_EEG, f"{out_name}_band_{band_names[band]}.mat"),
            {"BLP" if "BLP" in out_name else "EEG": data[:samples_for_chunk, :, :]},
        )
        if "EEG" in out_name:
            for i in range(2, 4):
                savemat(
                    os.path.join(
                        output_EEG, f"{out_name}_band_{band_names[band]}_ses{i}.mat"
                    ),
                    {
                        "EEG": data[
                            (i - 1) * samples_for_chunk : i * samples_for_chunk, :, :
                        ]
                    },
                )

In [ ]:
config_string = f"[EEG_bands]\nfile_path = {output_EEG}\nfile_name = EEG_124s_band_%(dataset_sub)s.mat\nfield_name = EEG\nhealthy_control_start = 0\nhealthy_control_end = 0\n"
config.read(config_ini)
if config.has_section("EEG_bands"):
    print(
        f"The config file '{config_ini}' already contains section 'EEG_bands'.\nCheck that it corresponds to:\n",
        config_string,
    )
else:
    with open(config_ini, "a") as fp:
        print(config_string, file=fp)
    print("'EEG_bands' added to config file.")


config_string = f"[EEG_bands_ses2]\nfile_path = {output_EEG}\nfile_name = EEG_124s_band_%(dataset_sub)s_ses2.mat\nfield_name = EEG\nhealthy_control_start = 0\nhealthy_control_end = 0\n"
config.read(config_ini)
if config.has_section("EEG_bands_ses2"):
    print(
        f"The config file '{config_ini}' already contains section 'EEG_bands'.\nCheck that it corresponds to:\n",
        config_string,
    )
else:
    with open(config_ini, "a") as fp:
        print(config_string, file=fp)
    print("'EEG_bands' added to config file.")


config_string = f"[EEG_bands_ses3]\nfile_path = {output_EEG}\nfile_name = EEG_124s_band_%(dataset_sub)s_ses3.mat\nfield_name = EEG\nhealthy_control_start = 0\nhealthy_control_end = 0\n"
config.read(config_ini)
if config.has_section("EEG_bands_ses3"):
    print(
        f"The config file '{config_ini}' already contains section 'EEG_bands'.\nCheck that it corresponds to:\n",
        config_string,
    )
else:
    with open(config_ini, "a") as fp:
        print(config_string, file=fp)
    print("'EEG_bands' added to config file.")


config_string = f"[electrodeBLP]\nfile_path = {output_EEG}\nfile_name = electrodeBLP_band_%(dataset_sub)s.mat\nfield_name = BLP\nhealthy_control_start = 0\nhealthy_control_end = 0\n"
config.read(config_ini)
if config.has_section("electrodeBLP"):
    print(
        f"The config file '{config_ini}' already contains section 'electrodeBLP'.\nCheck that it corresponds to:\n",
        config_string,
    )
else:
    with open(config_ini, "a") as fp:
        print(config_string, file=fp)
    print("'electrodeBLP' added to config file.")


config_string = f"[electrodeBLP_shadow]\nfile_path = {output_EEG}\nfile_name = electrodeBLP_shadow_band_%(dataset_sub)s.mat\nfield_name = BLP\nhealthy_control_start = 0\nhealthy_control_end = 0\n"
config.read(config_ini)
if config.has_section("electrodeBLP_shadow"):
    print(
        f"The config file '{config_ini}' already contains section 'electrodeBLP_shadow'.\nCheck that it corresponds to:\n",
        config_string,
    )
else:
    with open(config_ini, "a") as fp:
        print(config_string, file=fp)
    print("'electrodeBLP_shadow' added to config file.")

# iEEG

Download size: 32 GB

In [ ]:
# http://ieeg-swez.ethz.ch/long-term_dataset/
# A. Burrello, L. Cavigelli, K. Schindler, L. Benini, A. Rahimi, ‘‘Laelaps: An Energy-Efficient Seizure Detection Algorithm from Long-term Human iEEG Recordings without False Alarms’’ in proceedings of the ACM/IEEE Design, Automation, and Test in Europe Conference (DATE), Florence, Italy, March 25-29, 2019.
import io, re
from scipy import interpolate


baseURL = "http://ieeg-swez.ethz.ch/long-term_dataset/"
output_iEEG = os.path.join(MAIN_DATA_FOLDER, "iEEG")
if not os.path.isdir(output_iEEG):
    os.makedirs(output_iEEG)

csv_fname = f"iEEG_info.csv"
csv_path = os.path.join(output_iEEG, csv_fname)
appendix = "info.mat"
N_subj = 18
N = 4
tot_sec = 3600

In [ ]:
bracket = np.array([-2700, 0, 2700])
bracket2 = np.array([-600, 0, 1800])

infos = []
with open(os.path.join(output_iEEG, "dataSources.txt"), "w") as ds:
    for i in range(1, N_subj + 1):
        sub = f"ID{i:02}"
        fname = "_".join([sub, appendix])
        URL = os.path.join(baseURL, sub, fname)
        request_data = requests.get(URL)
        file_like = io.BytesIO(request_data.content)
        mat = loadmat(file_like)
        fs = mat["fs"][0, 0]
        bad_hours = set()
        bad_hours2 = set()
        for p in zip(mat["seizure_begin"], mat["seizure_end"]):
            for e in p:
                # the correct numbering is applied when downloading
                bad_hours.update(((bracket + e[0]) / 3600).astype(int))
                bad_hours2.update(((bracket2 + e[0]) / 3600).astype(int))
        if i in [15, 17]:
            # we manually identified these hours as containing seizures or artifacts
            bad_hours.add(0)
            bad_hours2.add(0)

        max_hour_request = requests.get(baseURL + f"ID{i:02}/")
        max_hour = 0
        for line in max_hour_request.content.decode().splitlines():
            match = re.findall(r"_(\d+)h", line)
            if match:
                max_hour = max(max_hour, int(match[0]))
        # the very last hour is often incomplete
        max_hour -= 1

        good_hours = set(range(max_hour)).difference(bad_hours)
        if len(good_hours) < 3:
            print("Not enough hours clear of seizures, using a smaller margin.")
            good_hours = set(range(max_hour)).difference(bad_hours2)

        first = min(good_hours)
        if max(good_hours) - first <= 48:
            second = min(
                good_hours, key=lambda x: np.abs(x - (max(good_hours) - first) / 2)
            )
            third = max(good_hours)
        else:
            second = min(good_hours, key=lambda x: np.abs(x - (24 + first)))
            third = min(good_hours, key=lambda x: np.abs(x - (24 + second)))

        for s, hour in enumerate([first, second, third], start=1):
            out_fname = f"iEEG_ID{i:02}_ses{s:02}.mat"
            out_path = os.path.join(output_iEEG, out_fname)
            data_fname = "_".join([sub, f"{hour+1}h.mat"])
            mat_path = os.path.join(baseURL, sub, data_fname)

            if not os.path.isfile(out_path):
                iEEG_file_response = requests.get(
                    mat_path,
                    stream=True,
                )

                with tqdm(
                    desc=f"Subj {i}, ses {s}, {hour} h", unit="B", unit_scale=True
                ) as progress_bar:
                    with open(out_path, "wb") as fp:
                        for data in iEEG_file_response.iter_content(block_size):
                            progress_bar.update(len(data))
                            fp.write(data)

            if s == 1:
                print(i, mat_path, good_hours, file=ds)
                mat = loadmat(out_path)
                electrodes = mat["EEG"].shape[0]
                samples = mat["EEG"].shape[1]

        infos.append([i, fs, electrodes, samples, first, second, third])

subj_data = pd.DataFrame(
    infos,
    columns=[
        "Subject",
        "fs_Hz",
        "Electrodes",
        "Sampling",
        "first_hour",
        "second_hour",
        "third_hour",
    ],
).set_index("Subject")
subj_data["Max_spikes"] = np.nan
subj_data["Tot_spikes"] = np.nan
subj_data["Sync_spikes"] = np.nan
subj_data.to_csv(csv_path, index=True)

In [ ]:
num_electrodes = subj_data.Electrodes.min()
if not os.path.isfile(os.path.join(output_iEEG, "usedElectrodes.json")):
    rs = np.random.RandomState(42)
    selected_electrodes = {}
    for i in range(1, 19):
        selected_electrodes[i] = np.sort(
            rs.choice(subj_data.loc[i, "Electrodes"], num_electrodes, False)
        ).tolist()

    with open(os.path.join(output_iEEG, "usedElectrodes.json"), "w") as fp:
        json.dump(selected_electrodes, fp)
else:
    with open(os.path.join(output_iEEG, "usedElectrodes.json"), "r") as fp:
        selected_electrodes = {int(k): v for k, v in json.load(fp).items()}

for session in range(1, 4):
    for band in bands:  #
        if not os.path.isfile(
            os.path.join(output_iEEG, f"iEEG_ses{session:02}_band_{band_names[band]}.mat")
        ):
            fs_dest = fs_band(band)
            tot_samp_dest = fs_dest * seconds_for_chunk
            new_dataset = np.empty([tot_samp_dest, num_electrodes, N_subj])
            for i in tqdm(range(1, N_subj + 1), desc=f"Session {session}, Band {band_names[band]}"):
                f_sample = subj_data.loc[i, "fs_Hz"]

                sos = signal.butter(N, bands[band], "bp", False, "sos", f_sample)
                out_fname = f"iEEG_ID{i:02}_ses{session:02}.mat"
                in_path = os.path.join(output_iEEG, out_fname)
                EEG = loadmat(in_path)["EEG"]

                start = (tot_sec - 5 * seconds_for_chunk) // 2 * f_sample
                stop = (tot_sec + 5 * seconds_for_chunk) // 2 * f_sample

                filtered = signal.sosfiltfilt(
                    sos, EEG[selected_electrodes[i], start:stop]
                )
                new_dataset[:, :, i - 1] = signal.resample(
                    filtered.T, 5 * tot_samp_dest
                )[2 * tot_samp_dest : 3 * tot_samp_dest, :]
            savemat(
                os.path.join(output_iEEG, f"iEEG_ses{session:02}_band_{band_names[band]}.mat"),
                {"iEEG": new_dataset},
            )

In [ ]:
def count_spikes(EEG: np.ndarray, fs_orig: int):
    """This function implements the IED detection algorithm from:
       R. Janca et al. (2015), "Detection of Interictal Epileptiform Discharges Using
       Signal Envelope Distribution Modelling: Application to Epileptic and Non-Epileptic
       Intracranial Recordings", Brain Topography, 28(1), 172-183.
       https://doi.org/10.1007/S10548-014-0379-1/FIGURES/7

       It does not include the removal of epochs with beta and mu rhythms that can lead
       to false positives (see the paper for more).

    Parameters
    ----------
    EEG : np.ndarray
        Array of the shape [#samples, #electrodes] containing the EEG data.
    fs_orig : int
        The sampling frequency of the EEG for correct filtering

    Returns
    -------
    int, int, int, list
        maximum number of spikes in a single electrode,
        total number of spikes in all electrodes,
        number of electrodes involved in the most widespread spike,
        a list containing, for each electrode, start time and duration of each spike
    """
    fs_dest = 200
    tollerance = round(0.12 * fs_dest)
    tot_samples_dest = round(EEG.shape[1] / fs_orig * fs_dest)

    sos = signal.butter(N, [10, 60], "bp", False, "sos", fs_orig)

    filtered = signal.sosfiltfilt(sos, EEG)
    resampled = signal.resample(filtered.T, tot_samples_dest)
    synchrony = []
    spike_data = []
    z = signal.hilbert(resampled[:, :], axis=0)
    all_power = np.absolute(z)
    for electrode in range(resampled.shape[1]):
        power = all_power[:, electrode]
        thresholds = []
        for window_start in range(0, resampled.shape[0] - fs_dest * 5, fs_dest):
            mu = np.mean(np.log(power[window_start : window_start + fs_dest * 5]))
            sigma = np.sqrt(
                np.sum(
                    (np.log(power[window_start : window_start + fs_dest * 5]) - mu) ** 2
                )
                / (fs_dest * 5 - 1)
            )
            mode_lognor = np.exp(mu - sigma**2)
            median_lognor = np.exp(mu)
            thresholds.append(3.65 * (mode_lognor + median_lognor))
        thresholds = [thresholds[0]] + thresholds + [thresholds[-1]]
        cs = interpolate.CubicSpline(
            np.concatenate(
                [
                    [0],
                    np.arange(0, resampled.shape[0] - fs_dest * 5, fs_dest)
                    + 2.5 * fs_dest,
                    [resampled.shape[0]],
                ]
            ),
            thresholds,
            bc_type="natural",
        )
        continuous_threshold = cs(np.arange(resampled.shape[0]))
        tmp = np.cumsum(continuous_threshold)
        tmp[fs_dest:] = tmp[fs_dest:] - tmp[:-fs_dest]
        mid_len = tmp[fs_dest - 1 :].shape[0]
        head = (resampled.shape[0] - mid_len) // 2
        tail = resampled.shape[0] - mid_len - head
        smooth_threshold = np.concatenate(
            [
                np.full(head, tmp[fs_dest - 1] / fs_dest),
                tmp[fs_dest - 1 :] / fs_dest,
                np.full(tail, tmp[-1] / fs_dest),
            ]
        )
        is_spike = (power > smooth_threshold).astype(int)
        is_spike[0] = 0
        is_spike[-1] = 0
        synchrony.append(is_spike[:])
        transitions = np.diff(is_spike)
        spike_begins = np.where(transitions == 1)[0]
        spike_ends = np.where(transitions == -1)[0]
        spike_times = []
        spike_durations = []
        for begin in spike_begins:
            if ((spike_ends > (begin - tollerance)) & (spike_ends < begin)).any():
                end = np.min(spike_ends[spike_ends > begin]) / fs_dest
                spike_durations[-1] = end - spike_times[-1]
            spike_times.append(begin / fs_dest)
            end = np.min(spike_ends[spike_ends > begin]) / fs_dest
            spike_durations.append(end - spike_times[-1])
        spike_data.append([spike_times, spike_durations])
    all_spik = np.stack(synchrony)
    syncsum = np.sum(all_spik, 0)
    spike_stat = [len(a[0]) for a in spike_data]
    return max(spike_stat), np.sum(spike_stat), max(syncsum), spike_data

In [ ]:
subj_data = pd.read_csv(csv_path, index_col=0)
with open(os.path.join(output_iEEG, "usedElectrodes.json"), "r") as fp:
    selected_electrodes = {int(k): v for k, v in json.load(fp).items()}
spike_data_all = {}
for i in tqdm(range(1, N_subj + 1), desc=f"Counting IEDs"):
    out_fname = f"iEEG_ID{i:02}_ses01.mat"
    in_path = os.path.join(output_iEEG, out_fname)
    EEG = loadmat(in_path)["EEG"]
    start = (tot_sec - seconds_for_chunk) // 2 * subj_data.loc[i, "fs_Hz"]
    durat = seconds_for_chunk * subj_data.loc[i, "fs_Hz"]
    max_spikes, tot_spikes, sync_spikes, spike_data = count_spikes(
        EEG[selected_electrodes[i], start : start + durat], subj_data.loc[i, "fs_Hz"]
    )
    subj_data.loc[i, ["Max_spikes", "Tot_spikes", "Sync_spikes"]] = (
        max_spikes,
        tot_spikes,
        sync_spikes,
    )
    spike_data_all[i] = {k: v for k, v in zip(selected_electrodes[i], spike_data)}

subj_data.to_csv(csv_path, index=True)
with open(os.path.join(output_iEEG, "spikes_by_subject_electrode.json"), "w") as fp:
    json.dump(spike_data_all, fp)

for session in range(1, 4):
    config_string = f"[iEEG_ses{session:02}]\nfile_path = {output_iEEG}\nfile_name = iEEG_ses{session:02}_band_%(dataset_sub)s.mat\nfield_name = iEEG\nhealthy_control_start = 0\nhealthy_control_end = 0\n"
    config.read(config_ini)
    if config.has_section(f"iEEG_ses{session:02}"):
        print(
            f"The config file '{config_ini}' already contains section 'iEEG_ses{session:02}'.\nCheck that it corresponds to:\n",
            config_string,
        )
    else:
        with open(config_ini, "a") as fp:
            print(config_string, file=fp)
        print(f"'iEEG_ses{session:02}' added to config file.")

In [ ]:
subj_data = pd.read_csv(csv_path, index_col=0)
num_electrodes = subj_data.Electrodes.min()
if not os.path.isfile(os.path.join(output_iEEG, "usedElectrodes.json")):
    raise FileNotFoundError("Generate the regular iEEG dataset first.")
else:
    with open(os.path.join(output_iEEG, "usedElectrodes.json"), "r") as fp:
        selected_electrodes = {int(k): v for k, v in json.load(fp).items()}


tot_samp_dest = 12276
for band in bands: 
    if not os.path.isfile(os.path.join(output_iEEG, f"iEEG_long_band_{band_names[band]}.mat")):
        fs_dest = fs_band(band)
        new_dataset = np.empty([tot_samp_dest, num_electrodes, N_subj])
        for i in tqdm(range(1, N_subj + 1), desc=f"Band {band_names[band]}"):
            f_sample = subj_data.loc[i, "fs_Hz"].astype(int)
            sec_orig = tot_samp_dest // fs_dest
            sos = signal.butter(N, bands[band], "bp", False, "sos", f_sample)
            out_fname = f"iEEG_ID{i:02}_ses01.mat"
            in_path = os.path.join(output_iEEG, out_fname)
            EEG = loadmat(in_path)["EEG"]

            start = (tot_sec - sec_orig - 20) // 2 * f_sample
            stop = (tot_sec + sec_orig + 20) // 2 * f_sample

            filtered = signal.sosfiltfilt(sos, EEG[selected_electrodes[i], start:stop])
            new_dataset[:, :, i - 1] = signal.resample(
                filtered.T, 20 * fs_dest + tot_samp_dest
            )[10 * fs_dest : 10 * fs_dest + tot_samp_dest, :]
        savemat(os.path.join(output_iEEG, f"iEEG_long_band_{band_names[band]}.mat"), {"iEEG": new_dataset})


config_string = f"[iEEG_long]\nfile_path = {output_iEEG}\nfile_name = iEEG_long_band_%(dataset_sub)s.mat\nfield_name = iEEG\nhealthy_control_start = 0\nhealthy_control_end = 0\n"
config.read(config_ini)
if config.has_section("iEEG_long"):
    print(
        f"The config file '{config_ini}' already contains section 'iEEG_long'.\nCheck that it corresponds to:\n",
        config_string,
    )
else:
    with open(config_ini, "a") as fp:
        print(config_string, file=fp)
    print("'iEEG_long' added to config file.")

In [ ]:
out_path = os.path.join(output_iEEG, "sampleSeizure.mat")
sub = f"ID12"
data_fname = "_".join([sub, f"176h.mat"])
mat_path = os.path.join(baseURL, sub, data_fname)
if not os.path.isfile(out_path):
    iEEG_file_response = requests.get(
        mat_path,
        stream=True,
    )

    with tqdm(desc=f"Subj 12, 176 h", unit="B", unit_scale=True) as progress_bar:
        with open(out_path, "wb") as fp:
            for data in iEEG_file_response.iter_content(block_size):
                progress_bar.update(len(data))
                fp.write(data)

sec_seizure = 183.936 + 150
f_sample = 1024
EEG = loadmat(out_path)["EEG"]

start = 1782743 - 51200 - 10240
stop = 1971102 + 102400 + 10240
for band in bands:  #
    sos = signal.butter(N, bands[band], "bp", False, "sos", f_sample)
    filtered = signal.sosfiltfilt(sos, EEG[:, start:stop], axis=1)[:, 10240:-10240]
    fs_dest = fs_band(band)
    sample_band = fs_dest * 45
    band_shift = sample_band // 10
    tot_samp_dest = int(fs_dest * sec_seizure)
    resampled = signal.resample(filtered.T, tot_samp_dest)

    N_blocks = (tot_samp_dest - sample_band) // band_shift
    new_dataset = np.empty([sample_band, EEG.shape[0], N_blocks])

    for i in tqdm(range(N_blocks), desc=f"Band {band_names[band]}"):
        new_dataset[:, :, i] = resampled[
            i * band_shift : i * band_shift + sample_band, :
        ]
    savemat(
        os.path.join(output_iEEG, f"sampleSeizure_band_{band_names[band]}.mat"), {"iEEG": new_dataset}
    )

power = np.absolute(signal.hilbert(EEG[:, start:stop])[:, 10240:-10240]).T
electrode_mean_power = np.mean(power, axis=1)
windowed_mean_power = np.mean(
    np.reshape(
        electrode_mean_power[: 64 * int(45 * 1024 / 10)], [-1, int(45 * 1024 / 10)]
    ),
    axis=1,
)
np.save(os.path.join(output_iEEG, f"sampleSeizure_power.npy"), windowed_mean_power)

config_string = f"[sampleSeizure]\nfile_path = {output_iEEG}\nfile_name = sampleSeizure_band_%(dataset_sub)s.mat\nfield_name = iEEG\nhealthy_control_start = 0\nhealthy_control_end = 0\n"
config.read(config_ini)
if config.has_section("sampleSeizure"):
    print(
        f"The config file '{config_ini}' already contains section 'sampleSeizure'.\nCheck that it corresponds to:\n",
        config_string,
    )
else:
    with open(config_ini, "a") as fp:
        print(config_string, file=fp)
    print("'sampleSeizure' added to config file.")

# Single units spikes

Download size: 40 GB

In [ ]:
from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache

In [ ]:
output_SUS = os.path.join(MAIN_DATA_FOLDER, "SUS")
if not os.path.isdir(output_SUS):
    os.makedirs(output_SUS)
DOWNLOAD_COMPLETE_DATASET = True
manifest_path = os.path.join(output_SUS, "manifest.json")
cache = EcephysProjectCache.from_warehouse(manifest=manifest_path)

# these are the session numbers that went through the "connectivity" experiment and have enough quality units
GOOD_SESSIONS = [
    831882777,
    816200189,
    771160300,
    786091066,
    779839471,
    778998620,
    781842082,
    778240327,
    793224716,
    839068429,
    794812542,
    768515987,
    767871931,
    840012044,
    766640955,
    847657808,
]

TIMES = [0.125 * 2**i for i in range(7)]

In [ ]:
def extract_groups_from_sessions(SESSION_NUMBER):
    session = cache.get_session_data(
        SESSION_NUMBER,
        amplitude_cutoff_maximum=0.08,
        presence_ratio_minimum=0.98,
        isi_violations_maximum=0.2,
    )
    suc = session.structurewise_unit_counts
    interesting_areas = [
        area for area in suc[suc >= 32].index.to_list() if area != "grey"
    ]

    selected = {}
    all_ind = []
    for structure in interesting_areas:
        units = session.units[
            session.units["ecephys_structure_acronym"] == structure
        ].copy()
        units["badness"] = (
            np.sqrt(
                4 * units.isi_violations**2
                + 2 * units.amplitude_cutoff**2
                + 3 * (1 - np.nan_to_num(units.nn_hit_rate)) ** 2
            )
            / 3
        )
        blocks = len(units) // 32
        units.sort_values("badness", inplace=True)
        units["good"] = True
        units.iloc[32 * blocks :, -1] = False
        units.sort_values(
            ["good", "probe_vertical_position"], ascending=False, inplace=True
        )
        selected[structure] = []
        for i in range(blocks):
            selected[structure].append(
                units[32 * i : 32 * (i + 1)]
                .sort_values(
                    "probe_vertical_position",
                    key=lambda x: np.abs(
                        x
                        - units[32 * i : 32 * (i + 1)].probe_vertical_position.median()
                    ),
                )
                .index
            )
            all_ind.append(selected[structure][-1].to_series())
        session.units[session.units["ecephys_structure_acronym"] != "grey"].loc[
            pd.concat(all_ind)
        ].to_csv(os.path.join(output_SUS, f"selected_units_data_{SESSION_NUMBER}.csv"))
    stimulus_tab = session.get_stimulus_table(include_detailed_parameters=True)
    spontaneous = (
        stimulus_tab[stimulus_tab.stimulus_name == "spontaneous"]
        .replace("null", np.nan)
        .dropna(axis=1, how="any")
    )
    start, end = spontaneous.loc[
        spontaneous.duration == spontaneous.duration.max(), ("start_time", "stop_time")
    ].values[0]
    spiking_times = {}
    for region in selected:
        spiking_times[region] = []
        for block in selected[region]:
            spiking_times[region].append({})
            for unit in block:
                # np.argmax(times>time_point) is the index of the first time>time_point
                spiking_times[region][-1][unit] = session.spike_times[unit][
                    np.argmax(session.spike_times[unit] > start) : np.argmax(
                        session.spike_times[unit] > end
                    )
                ]
    # remove head if no unit is spiking
    minima = []
    maxima = []
    tot_blocks = 0
    for region in spiking_times:
        tot_blocks += len(spiking_times[region])
        for block in spiking_times[region]:
            for unit in block:
                minima.append(block[unit][2])
                maxima.append(block[unit][-3])
    max_min = max(minima)
    min_max = min(maxima)

    all_spikes, STEP = np.linspace(max_min, min_max, 30000, retstep=True)
    print(f"Step: {STEP*100:.5} ms")
    all_units = [(region, b, unit) for region in spiking_times for (b, block) in enumerate(spiking_times[region]) for unit in block]
    all_fr = pd.DataFrame(index=pd.Index(all_spikes), columns=pd.MultiIndex.from_tuples(all_units, names=["region","block", "unit"]))

    for region, block, unit in all_units:
        these_spikes = spiking_times[region][block][unit]
        i = np.searchsorted(these_spikes,all_spikes, "right")
        alpha = (all_spikes-these_spikes[i-1])/(these_spikes[i]-these_spikes[i-1])
        all_fr.loc[:, (region, block, unit)]=(alpha*(these_spikes[i+1]-these_spikes[i])+(these_spikes[i]-these_spikes[i-1])+(1-alpha)*(these_spikes[i-1]-these_spikes[i-2]))/2
            

    dataset = np.empty((len(all_spikes), tot_blocks, 6))
    for I, i in zip(np.arange(6),np.power(2, np.arange(6), dtype=int)):
        for j in range(tot_blocks):
            dataset[:,j,I]=1/all_fr.iloc[:,32*j:32*j+i].values.mean(axis=1)
    savemat(
            os.path.join(
                output_SUS, f"spiking_{SESSION_NUMBER}.mat"
            ),
            {"spikes": dataset},
        )

In [ ]:
for session in tqdm(GOOD_SESSIONS, desc="Session"):
    extract_groups_from_sessions(session)
config_string = f"[spiking]\nfile_path = {output_SUS}\nfile_name = spiking_%(dataset_sub)s.mat\nfield_name = spikes\nhealthy_control_start = 0\nhealthy_control_end = 0\n"
config.read(config_ini)
if config.has_section(f"spiking"):
    print(
        f"The config file '{config_ini}' already contains section 'spiking'.\nCheck that it corresponds to:\n",
        config_string,
    )
else:
    with open(config_ini, "a") as fp:
        print(config_string, file=fp)
    print(f"'spiking' added to config file.")